# 01 — Thermal inactivation curves at 50 °CFits the Sadana–Henley two-parameter model$$A(t) = A_\infty + (A_0 - A_\infty)\,e^{-kt}$$to the four cytosine deaminase variants, and reports which summary statisticsare actually defined for each one.**Data provenance:** digitized from the published figure, not raw lab numbers.Activity values carry ~1–2 percentage points of digitization uncertainty. See`scripts/digitize_figure.py`.

In [ ]:
import sys, pathlibsys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))import numpy as np, pandas as pd, matplotlib.pyplot as pltfrom stability.kinetics import fit_variant, residual_activityfrom stability.sequences import verify_mutations, length_reportverify_mutations()print("sequences OK:", length_report())df = pd.read_csv("../data/raw/inactivation_50C_digitized.csv", comment="#")df.groupby("variant").size()

## 1. Fit each variant

In [ ]:
fits = []for variant in ["WT", "DOUBLE", "D314A", "S126R"]:    g = df[df.variant == variant].sort_values("time_min")    fits.append(fit_variant(g.time_min.values, g.activity_pct.values,                            variant=variant, n_boot=600, seed=1))table = pd.DataFrame([f.as_row() for f in fits])[    ["variant", "n_points", "k", "a_inf", "tau", "t_half", "k_identifiable"]]table.round(4)

In [ ]:
for f in fits:    kl, kh = f.k_ci; al, ah = f.a_inf_ci    print(f"{f.variant:7s}  k = {f.k:.3f} [{kl:.3f}, {kh:.3f}]"          f"   A_inf = {f.a_inf:.1f} [{al:.1f}, {ah:.1f}] %")    for n in f.notes:        print(f"         ! {n}")

## 2. What is and is not measurable hereThree things fall out of the confidence intervals, and all three matter beforeany model gets compared against these numbers.**`A_inf` is the robust observable.** Its interval is about ±2 points for everyvariant, while the variants span 22 % to 60 %. The ordering is safe.**`k` is identifiable only for WT and D314A.** For S126R and DOUBLE the decay isfaster than the first sampling point, so the bootstrap interval spans orders ofmagnitude. Those two rates are not measurements, they are artefacts of the grid.**`t_half` is undefined for WT** (the plateau sits above 50 %, the curve nevercrosses) and **undecidable for DOUBLE**, whose `A_inf` interval straddles 50 %.Ranking the four variants by half-life is therefore not possible with this data,regardless of which model you compare against.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))tt = np.linspace(0, 400, 500)for f in sorted(fits, key=lambda f: -f.a_inf):    g = df[df.variant == f.variant]    p = ax.scatter(g.time_min, g.activity_pct, s=30, zorder=3)    ax.plot(tt, residual_activity(tt, f.k, f.a_inf), color=p.get_facecolor()[0],            label=f"{f.variant}   A∞={f.a_inf:.0f}%   k={f.k:.3f}" +                  ("" if f.k_identifiable else " (k not identifiable)"))    ax.axhline(f.a_inf, color=p.get_facecolor()[0], lw=0.6, ls=":", alpha=0.7)ax.axhline(50, ls="--", c="k", lw=1)ax.text(398, 51.5, "50 % — t_half only exists below this line", ha="right", fontsize=8)ax.set_xlabel("Time at 50 °C (min)"); ax.set_ylabel("Residual activity (%)")ax.set_ylim(0, 105); ax.legend(fontsize=9); fig.tight_layout()fig.savefig("../docs/resultados/01-inactivation-fits.png", dpi=150)

## 3. Two observations worth taking to the lab**WT and D314A decay at the same rate.** Their fitted `k` are 0.108 and 0.109 andtheir intervals overlap almost completely. What differs is only the plateau:60 % vs 47 %. D314A does not change how fast the enzyme inactivates, it changeswhat fraction survives. Those are different physical claims.**The double mutant behaves like D314A alone, not like the sum.** S126R on itsown collapses to 22 %, but S126R combined with D314A sits at 48 %, within errorof D314A alone at 47 %. The effects are not additive — D314A appears to suppresswhat S126R does. If this holds in the raw data, it is epistasis, and noTm-predicting model has any route to it.Both of these rest on digitized points, so they are hypotheses to confirm againstthe raw numbers, not results.

In [ ]:
summary = table.copy()summary.to_csv("../data/processed/inactivation_fits.csv", index=False)summary